---
# Homework 1: Multilayer Perceptron (MLP)   (100 points)

In this homework, we're going to build a multilayer perceptron (MLP) model and then train it on an image classification dataset.

In [1]:
# import core PyTorch modules for building and training neural networks
import torch
from torch import nn
# import torchvision for image transforms
import torchvision
import torchvision.transforms as transforms
# import Dataset and DataLoader utilities for handling data
from torch.utils.data import Dataset, DataLoader
# plotting library (for visualization)
import matplotlib.pyplot as plt
# progress bar utility
from tqdm import tqdm
# used to evaluate classification performance
from sklearn.metrics import classification_report
# used for splitting datasets if needed
from sklearn.model_selection import train_test_split
# data handling libraries
import pandas as pd
import numpy as np
# used to track training time
import time
# used to save best model without reference issues
import copy
from mads.lib.path import assets
torch.set_num_threads(4)
torch.set_num_interop_threads(4)

# Loading Data (0 points)

This is an example of building a torch data loader for a dataset. Here we have used it to load the Fashion MNIST image dataset. No code needed here.

The raw data is downloaded from: [Kaggle - Fashion MNIST Dataset](https://www.kaggle.com/datasets/zalando-research/fashionmnist?resource=download)

We have already built training, validation, and test datasets for this homework. 

In [2]:
"""Load the training, validation, and test csv files."""

# find file paths form course assets folder
train_file = assets.find("fashion_mnist/train.csv")
test_file = assets.find("fashion_mnist/test.csv")
valid_file = assets.find("fashion_mnist/val.csv")

# read each csv file into pandas Dataframe
train_csv = pd.read_csv(train_file)
test_csv = pd.read_csv(test_file)
valid_csv = pd.read_csv(valid_file)

In [3]:
"""Create custom PyTorch Dataset class."""

class FashionDataset(Dataset):
    """User defined class to build a datset using Pytorch class Dataset."""
    
    def __init__(self, data, transform = None):
        """Initialize the dataset and preprocess labels/images""" 

        # convert DataFrame values into a list for easier interation
        self.fashion_MNIST = list(data.values)

        # store any optional transform (such as ToTensor)
        self.transform = transform

        # lists to hold labels and image pixel values separately
        label = []
        image = []

        # each row contains the label in the first column
        # and pixel values in the remaining columns
        for i in self.fashion_MNIST:
             # first column is of labels.
            label.append(i[0])  # first value is the class label
            image.append(i[1:]) # remaining values are image pixels
        
        # convert label to a NumPy array    
        self.labels = np.asarray(label)
        # Dimension of Images = 28 * 28 * 1. where height = width = 28 and color_channels = 1.
        self.images = np.asarray(image).reshape(-1, 28, 28, 1).astype('float32')
        self.images = self.images/256

    def __getitem__(self, index):
        """Return one image-label pair at the given index."""

        # get label and image at the selected index
        label = self.labels[index]
        image = self.images[index]

        # apply transform if one is provided
        if self.transform is not None:
            image = self.transform(image)

        return image, label

    def __len__(self):
        """Return the total number of examples in the dataset."""
        return len(self.images)

In [4]:
"""Create datasets and dataloaders."""

# number of samples per batch during training/evaluation
batch_size=256

# create dataset objects and convert images to PyTorch tensors
train_set = FashionDataset(train_csv, transform=transforms.Compose([transforms.ToTensor()]))
val_set = FashionDataset(valid_csv, transform=transforms.Compose([transforms.ToTensor()]))
test_set = FashionDataset(test_csv, transform=transforms.Compose([transforms.ToTensor()]))

# wrap datasets in DataLoaders for batch processing
train_loader = DataLoader(train_set, batch_size=batch_size)
val_loader = DataLoader(val_set, batch_size=batch_size)
test_loader = DataLoader(test_set, batch_size=batch_size)

# Question 1: Build a MLP (50 pts)
In this part, you will build a multilayer perceptron (MLP) neural network.

The input to the model initialization should include the number of inputs, the number of outputs and the size of the hidden layer. 

In the image classification dataset we use (Fashion MNIST), all images are 28*28=784, so the number of inputs should always be 784. The dataset has 10 labels (classes), so the number of outputs should be always 10. The size of a hidden layer is a tunable hypeerparameter.

For a fair comparison, we ask you to implement exactly two perceptron layers in the MLP. Please ensure that the hidden layer size is consistent to connect the two perceptron layers. Use ReLU as the activation function. 

A simple trick here is the initialization. A good initialization is always helpful to the model performance.

In [5]:
"""Define the MLP model."""

class MLP(nn.Module):
    def __init__(self,num_inputs=784,num_outputs=10,num_hiddens=256):
        super(MLP, self).__init__()

        # flatten the 28x28 image into a 784-length vector
        self.flatten = nn.Flatten()
        
        # first fully connected layer:
        self.fc1 = nn.Linear(num_inputs, num_hiddens)

        # ReLU activation introduces nonlinearity
        self.relu = nn.ReLU()

        # second fully connected layer:
        # hidden layer -> output layer with 10 classes
        self.fc2 = nn.Linear(num_hiddens, num_outputs)

        # Xavier/Glorot initialization for weights
        # helps stabilize gradients in deep networks
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.zeros_(self.fc1.bias)
        nn.init.xavier_uniform_(self.fc2.weight)
        nn.init.zeros_(self.fc2.bias)

    def forward(self, X):
        """Define how input flows through the network."""

        # convert image tensor from shape [batch, 1, 28, 28]
        # to [batch, 784]
        X = self.flatten(X)

        # pass through first linear layer
        X = self.fc1(X)

        # apply ReLU activation
        X = self.relu(X)

        # pass through final output layer
        X = self.fc2(X)

        # return raw logits (CrossEntropyLoss applies softmax internally)
        return X
        

In [6]:
"""Define hyperparameters."""

# candidate hidden layer sizes
hiddens=[256,512,1024]

# candidate learning rates
lrs=[5e-2, 1e-1, 5e-1]

# input dimension: 28 x 28 pixels flattened
num_inputs=784

# output dimension: 10 Fashion MNIST classes
num_outputs=10

# loss function for multi-class classification
loss_function = nn.CrossEntropyLoss()

In [7]:
# function for evaluating the model performance
def eval_model(model,data_loader):
    model.eval()
    y_true_list=[]
    y_pred_list=[]
    model.eval()
    for x,y in data_loader:
        outputs=model(x)
        _, y_pred = torch.max(outputs, 1)
        y_pred_list.extend(y_pred.clone().detach().tolist())
        y_true_list.extend(y.clone().detach().tolist())
    acc=classification_report(y_true_list, y_pred_list,output_dict=True)['accuracy']
    return acc

In [8]:
# Hidden tests in this cell

# Question 2: Train the MLP (50 pts)
In this part, you will write a few lines to train the multilayer perceptron neural network you just built.

Here's the list of things that you need to implement. All of them can (should) be done using one line of code.

    Initialize the model with a set of hyperparameters
    Initialize the optimizer with the model's trainable parameters
    Set the model into the training mode
    For every batch of data: 
        zero the gradient in the optimizer
        feed the input into the model
        compute the loss
        back propagate the loss
        update the optimizer

* Every 5 epochs evaluate your model on the test set
    * Keep track of the highest accuracy achieved on the test set using variable `current_best`
    * Your best performing model should be stored in variable ```best_model``` (**note**: you will want to use ```copy.deepcopy``` when copying your best model to avoid referencing the wrong model object)


In [9]:
random_seed = 3407
torch.manual_seed(random_seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [10]:
"""Train models across different hyperparameter settings."""

# record the starting time for time-limited training
start_time=time.time()

# keep track of best test accuracy seen so far
current_best=0.0

# store best model found
best_model=None

# loop over all hidden layer sizes
for h in hiddens:
    # loop over all learning rates
    for lr in lrs:
        
        # initialize model with current hyperparameters
        model = MLP(num_inputs = num_inputs, num_outputs = num_outputs, num_hiddens = h)
        
        # initialize Adam optimizer with the current learning rate
        optimizer = torch.optim.Adam(model.parameters(), lr = lr * 0.01)
        
        # train for up to 30 epochs
        for i in range(30):
            
            # Stop early if:
            # 1. accuracy reaches at least 0.85
            # 2. training time exceeds 15 minutes
            if current_best >= 0.85 or ((time.time()-start_time)/60) >15:
                break
                
            # set the model to training mode
            model.train()
            
            # iterate through mini-batches of training data
            for x,y in train_loader:
                
                # zero gradients
                optimizer.zero_grad()
                
                # forward pass: compute predicted outputs
                outputs = model(x)
                
                # compute loss between predictions and true labels
                loss = loss_function(outputs, y)
                
                # backward pass: compute gradients
                loss.backward()
                
                # update model weights
                optimizer.step()

            # evaluate the model every 5 epochs
            if (i + 1) % 5 == 0:
                acc = eval_model(model, test_loader)
                
                # save model if it is the best seen so far
                if acc > current_best:
                    current_best = acc
                    best_model = copy.deepcopy(model)


In [11]:
# Hidden tests in this cell

In [12]:
# Hidden tests in this cell